In [ ]:
import os
import requests
import pandas as pd
import datetime
from dotenv import load_dotenv

# 1. Identity & Path Setup
load_dotenv("../../config/.env")
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

PREFIX_MAP_FILE = "../../config/prefix_map.csv"
MASTER_FILE = "../../data/raw/metadata_ingest_undeduped.csv"

# 2. HL7 Specific Configuration
# We define the sources and their SSSOM-ready prefixes
HL7_SOURCES = {
    "HL7v3": {
        "url": "http://terminology.hl7.org/CodeSystem-v3-ReligiousAffiliation",
        "name": "HL7 v3 Religious Affiliation",
        "base_uri": "http://terminology.hl7.org/CodeSystem/v3-ReligiousAffiliation"
    },
    "HL7v2": {
        "url": "http://terminology.hl7.org/CodeSystem-v2-0006",
        "name": "HL7 v2 Table 0006",
        "base_uri": "http://terminology.hl7.org/CodeSystem/v2-0006"
    }
}

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/fhir+json'
}

COLUMN_ORDER = [
    "Source_System", "Primary_Label", "CURIE", "Formal_Label", 
    "Hierarchy_Path", "Synonyms", "Description", "Parent_IDs", 
    "Concept_ID", "URI", "Extraction_Date"
]

# 3. Update Prefix Map for both HL7 versions
def update_prefix_map(prefix, base_uri, source_name):
    new_entry = pd.DataFrame([{'Prefix': prefix, 'Base_URI': base_uri, 'Source_Name': source_name}])
    if os.path.exists(PREFIX_MAP_FILE):
        pmap = pd.read_csv(PREFIX_MAP_FILE)
        if prefix not in pmap['Prefix'].values:
            pmap = pd.concat([pmap, new_entry], ignore_index=True)
            pmap.to_csv(PREFIX_MAP_FILE, index=False)
    else:
        new_entry.to_csv(PREFIX_MAP_FILE, index=False)

for pref, info in HL7_SOURCES.items():
    update_prefix_map(pref, info['base_uri'], info['name'])

print("Cell 1: HL7 environment and Prefix Map initialized for v2 and v3.")

In [ ]:
def process_hl7_item(item, prefix, base_uri, parent_id=None, parent_path=""):
    """Recursively parses FHIR expansion items or compose concepts."""
    rows = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # HL7 uses 'code' for the ID and 'display' for the label
    code = str(item.get('code', ''))
    display = item.get('display', item.get('definition', 'Unknown'))
    system = item.get('system', base_uri)
    
    if not code: return [] # Skip if no code found

    current_path = f"{parent_path} > {display}" if parent_path else display
    
    row = {
        'Source_System': f"HL7 {prefix}",
        'Primary_Label': display,
        'CURIE': f"{prefix}:{code}",
        'Formal_Label': display,
        'Hierarchy_Path': current_path,
        'Synonyms': "", 
        'Description': f"HL7 System: {system}",
        'Parent_IDs': parent_id if parent_id else "",
        'Concept_ID': code,
        'URI': f"{system}#{code}",
        'Extraction_Date': timestamp
    }
    rows.append(row)

    # Check for 'contains' (Expansion) or 'concept' (Compose/Hierarchy)
    children = item.get('contains', item.get('concept', []))
    for sub_item in children:
        rows.extend(process_hl7_item(sub_item, prefix, base_uri, code, current_path))
            
    return rows

# --- Main Execution ---
for prefix, info in HL7_SOURCES.items():
    print(f"\n--- Harvesting {info['name']} ({prefix}) ---")
    
    # We append .json to the CodeSystem URL
    url = f"{info['url']}.json"
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        all_rows = []
        
        # CodeSystems store concepts in a top-level 'concept' array
        concepts = data.get('concept', [])
        
        # If not there, check if it's nested in a ValueSet expansion style (rare for CS)
        if not concepts:
            concepts = data.get('expansion', {}).get('contains', [])

        for item in concepts:
            all_rows.extend(process_hl7_item(item, prefix, info['base_uri']))
            
        if not all_rows:
            print(f"Warning: No concepts found for {prefix}. Check {url}")
            continue

        print(f"Captured {len(all_rows)} concepts.")
        
        # Save to MASTER_FILE
        df = pd.DataFrame(all_rows)
        for col in COLUMN_ORDER:
            if col not in df.columns:
                df[col] = ""
        
        df = df[COLUMN_ORDER]
        file_exists = os.path.isfile(MASTER_FILE)
        df.to_csv(MASTER_FILE, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')

    except Exception as e:
        print(f"Error harvesting {prefix}: {e}")

print(f"\nDone! HL7 data process complete.")